In [1]:
from set_seed_utils import set_random_seed
import os
import random
import numpy as np
import pickle
import torch
from tqdm import tqdm
from torch.utils.data import DataLoader
from token_utils_rep import EHRTokenizer
from dataset_utils_rep import HBERTFinetuneEHRDataset, batcher, UniqueIDSampler
from HEART_rep import HBERT_Finetune
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, auc, precision_recall_curve, precision_recall_fscore_support
import pandas as pd

Disabling PyTorch because PyTorch >= 2.1 is required but found 1.13.1
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [3]:
PHENO_ORDER = [
    "Acute and unspecified renal failure",
    "Acute cerebrovascular disease",
    "Acute myocardial infarction",
    "Cardiac dysrhythmias",
    "Chronic kidney disease",
    "Chronic obstructive pulmonary disease",
    "Conduction disorders",
    "Congestive heart failure; nonhypertensive",
    "Coronary atherosclerosis and related",
    "Disorders of lipid metabolism",
    "Essential hypertension",
    "Fluid and electrolyte disorders",
    "Gastrointestinal hemorrhage",
    "Hypertension with complications",
    "Other liver diseases",
    "Other lower respiratory disease",
    "Pneumonia",
    "Septicemia (except in labor)",
]

In [4]:
@torch.no_grad()
def evaluate(model, 
             dataloader, 
             device, 
             long_seq_idx=None, 
             task_type="binary", 
             subgroup_labels=None):
    """
    subgroup_labels: None 或 pandas.DataFrame / Series，长度必须等于 dataloader 总样本数，
                     每列为一个 0/1 subgroup（如 DIABETES/HF/...），仅在 binary 任务下使用。
    返回：
        all_performance:     overall 指标
        subset_performance:  long_seq 子集指标（若 long_seq_idx 不为 None，否则为 None）
        subgroup_performance: dict[subgroup_name -> metrics_dict] 或 None
    """
    model.eval()
    predicted_scores, gt_labels = [], []

    # 推理：收集 logits 与 labels
    for _, batch in enumerate(tqdm(dataloader, desc="Running inference")):
        batch = [x.to(device) if isinstance(x, torch.Tensor) else x for x in batch]
        labels = batch[-1]
        output_logits = model(*batch[:-1])
        predicted_scores.append(output_logits)
        gt_labels.append(labels)

    # ============= 二分类任务 ============= #
    if task_type == "binary":
        logits_all = torch.cat(predicted_scores, dim=0).view(-1)           # [N]
        labels_all = torch.cat(gt_labels, dim=0).view(-1).cpu().numpy()    # [N]
        scores_all = logits_all.cpu().numpy()
        ypred_all  = (logits_all > 0).float().cpu().numpy()

        tp = (ypred_all * labels_all).sum()
        precision = tp / (ypred_all.sum() + 1e-8)
        recall    = tp / (labels_all.sum() + 1e-8)
        f1        = 2 * precision * recall / (precision + recall + 1e-8)
        roc_auc   = roc_auc_score(labels_all, scores_all)
        prec_curve, rec_curve, _ = precision_recall_curve(labels_all, scores_all)
        pr_auc    = auc(rec_curve, prec_curve)

        all_performance = {
            "precision": float(precision),
            "recall":    float(recall),
            "f1":        float(f1),
            "auc":       float(roc_auc),
            "prauc":     float(pr_auc),
        }

        # ---- long_seq 子集 ----
        subset_performance = None
        if long_seq_idx is not None:
            idx = torch.as_tensor(long_seq_idx, device=logits_all.device, dtype=torch.long)
            logits_sub = logits_all.index_select(0, idx).view(-1)
            labels_sub = torch.as_tensor(labels_all, device=logits_all.device)[idx].cpu().numpy()
            scores_sub = logits_sub.cpu().numpy()
            ypred_sub  = (logits_sub > 0).float().cpu().numpy()

            tp = (ypred_sub * labels_sub).sum()
            precision = tp / (ypred_sub.sum() + 1e-8)
            recall    = tp / (labels_sub.sum() + 1e-8)
            f1        = 2 * precision * recall / (precision + recall + 1e-8)
            roc_auc   = roc_auc_score(labels_sub, scores_sub)
            prec_curve, rec_curve, _ = precision_recall_curve(labels_sub, scores_sub)
            pr_auc    = auc(rec_curve, prec_curve)

            subset_performance = {
                "precision": float(precision),
                "recall":    float(recall),
                "f1":        float(f1),
                "auc":       float(roc_auc),
                "prauc":     float(pr_auc),
            }

        # ---- subgroup analysis（仅 binary）----
        subgroup_performance = None
        if subgroup_labels is not None:
            import pandas as pd
            subgroup_performance = {}

            if isinstance(subgroup_labels, pd.Series):
                subgroup_df = subgroup_labels.to_frame()
            else:
                subgroup_df = subgroup_labels

            if len(subgroup_df) != logits_all.shape[0]:
                raise ValueError(
                    f"subgroup_labels 行数 {len(subgroup_df)} 与样本数 {logits_all.shape[0]} 不一致"
                )

            for col in subgroup_df.columns:
                mask_np = subgroup_df[col].to_numpy().astype(bool)
                if mask_np.sum() == 0:
                    continue  # 这个 subgroup 没有样本，跳过

                idx = torch.as_tensor(
                    np.where(mask_np)[0],
                    device=logits_all.device,
                    dtype=torch.long,
                )

                logits_sub = logits_all.index_select(0, idx).view(-1)
                labels_sub = torch.as_tensor(labels_all, device=logits_all.device)[idx].cpu().numpy()
                scores_sub = logits_sub.cpu().numpy()
                ypred_sub  = (logits_sub > 0).float().cpu().numpy()

                tp = (ypred_sub * labels_sub).sum()
                precision = tp / (ypred_sub.sum() + 1e-8)
                recall    = tp / (labels_sub.sum() + 1e-8)
                f1        = 2 * precision * recall / (precision + recall + 1e-8)
                roc_auc   = roc_auc_score(labels_sub, scores_sub)
                prec_curve, rec_curve, _ = precision_recall_curve(labels_sub, scores_sub)
                pr_auc    = auc(rec_curve, prec_curve)

                subgroup_performance[col] = {
                    "precision": float(precision),
                    "recall":    float(recall),
                    "f1":        float(f1),
                    "auc":       float(roc_auc),
                    "prauc":     float(pr_auc),
                }

        return all_performance, subset_performance, subgroup_performance

    # ============= Multi-label 任务 ============= #
    else:
        logits_all = torch.cat(predicted_scores, dim=0)    # [B, C]
        labels_all_t = torch.cat(gt_labels, dim=0)         # [B, C]

        def _compute_metrics(logits_sub, labels_sub):
            if logits_sub.device.type == "cpu" and logits_sub.dtype == torch.float16:
                prob_t = torch.sigmoid(logits_sub.float())
            else:
                prob_t = torch.sigmoid(logits_sub)

            ypred_t = (logits_sub > 0).to(torch.int32)

            y_true = labels_sub.cpu().numpy().astype(np.int32)
            y_pred = ypred_t.cpu().numpy().astype(np.int32)
            scores = prob_t.cpu().numpy()

            p_cls, r_cls, f1_cls, _ = precision_recall_fscore_support(
                y_true, y_pred, average=None, zero_division=0
            )

            C = y_true.shape[1]
            aucs, praucs = [], []
            for c in range(C):
                yt, ys = y_true[:, c], scores[:, c]
                if yt.max() == yt.min():
                    aucs.append(np.nan)
                    praucs.append(np.nan)
                else:
                    aucs.append(roc_auc_score(yt, ys))
                    prec_curve, rec_curve, _ = precision_recall_curve(yt, ys)
                    praucs.append(auc(rec_curve, prec_curve))

            summary = {
                "precision": float(np.mean(p_cls)),
                "recall":    float(np.mean(r_cls)),
                "f1":        float(np.mean(f1_cls)),
                "auc":       float(np.nanmean(aucs)) if np.any(~np.isnan(aucs)) else float("nan"),
                "prauc":     float(np.nanmean(praucs)) if np.any(~np.isnan(praucs)) else float("nan"),
            }

            per_class_df = pd.DataFrame({
                "precision": p_cls,
                "recall":    r_cls,
                "f1":        f1_cls,
                "auc":       aucs,
                "prauc":     praucs,
            }, index=PHENO_ORDER)

            return {"global": summary, "per_class": per_class_df}

        all_performance = _compute_metrics(logits_all, labels_all_t)

        subset_performance = None
        if long_seq_idx is not None:
            idx = torch.as_tensor(long_seq_idx, device=logits_all.device, dtype=torch.long)
            subset_performance = _compute_metrics(
                logits_all.index_select(0, idx),
                labels_all_t.index_select(0, idx)
            )

        # multi-label 不做 subgroup，统一返回 None
        subgroup_performance = None
        return all_performance, subset_performance, subgroup_performance

In [5]:
args = {
    "seed": 0,
    "dataset": "MIMIC-III", 
    "task": "stay",  # options: death, stay, readmission, next_diag_6m, next_diag_12m
    "encoder": "hi",  # options: hi_edge, hi_node, hi_edge_node
    "batch_size": 4,
    "eval_batch_size": 4,
    "pretrain_mask_rate": 0.7,
    "lr": 1e-4,
    "epochs": 500,
    "num_hidden_layers": 5,
    "num_attention_heads": 6,
    "attention_probs_dropout_prob": 0.2,
    "hidden_dropout_prob": 0.2,
    "edge_hidden_size": 32,
    "hidden_size": 288,  # must be divisible by num_attention_heads
    "intermediate_size": 288,
    "save_model": True,
    "gat": "None",
    "gnn_n_heads": 1,
    "gnn_temp": 1,
    "diag_med_emb": "simple",  # simple, tree
    "early_stop_patience": 5,
}

In [6]:
exp_name = "Pretrain-HBERT" \
    + "-" + str(args["dataset"]) \
    + "-" + str(args["encoder"]) \
    + "-" + str(args["pretrain_mask_rate"]) \
    + "-" + str(args["hidden_size"]) \
    + "-" + str(args["edge_hidden_size"]) \
    + "-" + str(args["num_hidden_layers"]) \
    + "-" + str(args["num_attention_heads"]) \
    + "-" + str(args["attention_probs_dropout_prob"]) \
    + "-" + str(args["hidden_dropout_prob"]) \
    + "-" + str(args["intermediate_size"]) \
    + "-" + str(args["gat"]) \
    + "-" + str(args["gnn_n_heads"]) \
    + "-" + str(args["gnn_temp"]) \
    + "-" + str(args["diag_med_emb"])
print(exp_name)

Pretrain-HBERT-MIMIC-III-hi-0.7-288-32-5-6-0.2-0.2-288-None-1-1-simple


In [7]:
pretrained_weight_path = "./pretrained_models/" + exp_name + f"/pretrained_model.pt"
finetune_exp_name = f"Finetune-{args['task']}-" + exp_name
save_path = "./saved_model/" + finetune_exp_name
if args["save_model"] and not os.path.exists(save_path):
    os.makedirs(save_path)

In [8]:
args["predicted_token_type"] = ["diag"]
args["special_tokens"] = ("[PAD]", "[CLS]", "[SEP]", "[MASK0]")
args["max_visit_size"] = 15

full_data_path = f"/home/lideyi/HeteroGT-cuda/data_process/{args['dataset']}-processed/mimic.pkl"

if args["task"] == "next_diag_6m":
    finetune_data_path = f"/home/lideyi/HeteroGT-cuda/data_process/{args['dataset']}-processed/mimic_nextdiag_6m.pkl"
elif args["task"] == "next_diag_12m":
    finetune_data_path = f"/home/lideyi/HeteroGT-cuda/data_process/{args['dataset']}-processed/mimic_nextdiag_12m.pkl"
else:
    finetune_data_path = f"/home/lideyi/HeteroGT-cuda/data_process/{args['dataset']}-processed/mimic_downstream.pkl"

In [9]:
ehr_data = pickle.load(open(full_data_path, 'rb'))
diag_sentences = ehr_data["ICD9_CODE"].values.tolist()
gender_set = [["M"], ["F"]]
age_gender_set = [[str(c) + "_" + gender] for c in set(ehr_data["AGE"].values.tolist()) for gender in ["M", "F"]]
age_set = [[c] for c in set(ehr_data["AGE"].values.tolist())]    

In [10]:
tokenizer = EHRTokenizer(diag_sentences, gender_set, age_set, age_gender_set, special_tokens=args["special_tokens"])

In [11]:
train_data, val_data, test_data = pickle.load(open(finetune_data_path, 'rb'))

subgroup_names = ["DIABETES", "HYPERTENSION", "CKD", "HEART_FAILURE", "CAD", "COPD", "LIVER_DISEASE", "CANCER"]
val_subgroup_labels = val_data[subgroup_names].copy()
test_subgroup_labels = test_data[subgroup_names].copy()

In [12]:
train_dataset = HBERTFinetuneEHRDataset(
    train_data, tokenizer, 
    token_type=args["predicted_token_type"], 
    task=args["task"]
)

val_dataset = HBERTFinetuneEHRDataset(
    val_data, tokenizer, 
    token_type=args["predicted_token_type"], 
    task=args["task"]
)

test_dataset = HBERTFinetuneEHRDataset(
    test_data, tokenizer, 
    token_type=args["predicted_token_type"], 
    task=args["task"]
)

print(len(train_dataset), len(val_dataset), len(test_dataset))

train_dataloader = DataLoader(
    train_dataset, 
    batch_sampler=UniqueIDSampler(train_dataset.get_ids(), batch_size=args["batch_size"]),
    collate_fn=batcher(pad_id=tokenizer.vocab.word2id["[PAD]"], is_train=False), 
)

val_dataloader = DataLoader(
    val_dataset, 
    batch_sampler=UniqueIDSampler(val_dataset.get_ids(), batch_size=args["batch_size"]),
    collate_fn=batcher(pad_id=tokenizer.vocab.word2id["[PAD]"], is_train=False), 
)

test_dataloader = DataLoader(
    test_dataset, 
    batch_sampler=UniqueIDSampler(test_dataset.get_ids(), batch_size=args["eval_batch_size"]),
    collate_fn=batcher(pad_id=tokenizer.vocab.word2id["[PAD]"], is_train=False),
)

3153 6266 6358


In [13]:
long_adm_seq_crite = 3
val_long_seq_idx, test_long_seq_idx = [], []
for i in range(len(val_dataset)):
    hadm_id = list(val_dataset.records.keys())[i]
    num_adms = len(val_dataset.records[hadm_id])
    if num_adms >= long_adm_seq_crite:
        val_long_seq_idx.append(i)
for i in range(len(test_dataset)):
    hadm_id = list(test_dataset.records.keys())[i]
    num_adms = len(test_dataset.records[hadm_id])
    if num_adms >= long_adm_seq_crite:
        test_long_seq_idx.append(i)
print(len(val_long_seq_idx), len(test_long_seq_idx))

777 861


In [14]:
# examine a batch
batch = next(iter(train_dataloader))  # 取第一个 batch
input_ids, input_types, edge_index, visit_positions, labeled_batch_idx, labels = batch

# 打印每个张量的形状
print("input_ids shape:", input_ids.shape)
print("input_types shape:", input_types.shape)
print("visit_positions shape:", visit_positions.shape)
print("labeled_batch_idx shape:", len(labeled_batch_idx)) # it is a list
print("labels shape:", labels.shape)

input_ids shape: torch.Size([8, 22])
input_types shape: torch.Size([8, 22])
visit_positions shape: torch.Size([8])
labeled_batch_idx shape: 4
labels shape: torch.Size([4, 1])


In [15]:
args["vocab_size"] = len(args["special_tokens"]) + \
                     len(tokenizer.diag_voc.id2word) + \
                     len(tokenizer.age_voc.id2word) + \
                     len(tokenizer.gender_voc.id2word) + \
                     len(tokenizer.age_gender_voc.id2word)
args["label_vocab_size"] = 18  # only for diagnosis

In [16]:
if args["task"] in ["death", "stay", "readmission"]:
    eval_metric = "f1"
    task_type = "binary"
    loss_fn = F.binary_cross_entropy_with_logits
else:
    eval_metric = "prauc"
    task_type = "l2r"
    loss_fn = lambda x, y: F.binary_cross_entropy_with_logits(x, y)

In [17]:
def train_with_early_stopping(model, 
                              train_dataloader, 
                              val_dataloader, 
                              test_dataloader,
                              optimizer, 
                              loss_fn, 
                              device, 
                              args,
                              val_long_seq_idx = None,
                              test_long_seq_idx = None,
                              task_type="binary", 
                              eval_metric="f1",
                              val_subgroup_labels=None,
                              test_subgroup_labels=None):
    best_score = 0.
    best_val_metric = None
    best_test_metric = None
    best_test_long_seq_metric = None
    best_val_subgroup_metrics = None
    best_test_subgroup_metrics = None
    epochs_no_improve = 0

    for epoch in range(1, 1 + args["epochs"]):
        model.train()
        ave_loss = 0.

        for step, batch in enumerate(tqdm(train_dataloader, desc="Training Batches")):
            batch = [x.to(device) if isinstance(x, torch.Tensor) else x for x in batch]

            labels = batch[-1].float()
            output_logits = model(*batch[:-1])
            
            loss = loss_fn(output_logits.view(-1), labels.view(-1))
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            ave_loss += loss.item()

        ave_loss /= (step + 1)

        # ===== Evaluation（带 subgroup） =====
        val_metric, val_long_seq_metric, val_subgroup_metrics = evaluate(
            model, 
            val_dataloader, 
            device, 
            long_seq_idx=val_long_seq_idx, 
            task_type=task_type,
            subgroup_labels=val_subgroup_labels,
        )
        test_metric, test_long_seq_metric, test_subgroup_metrics = evaluate(
            model, 
            test_dataloader, 
            device, 
            long_seq_idx=test_long_seq_idx, 
            task_type=task_type,
            subgroup_labels=test_subgroup_labels,
        )

        if task_type != "binary":
            val_per_class_df = val_metric["per_class"]
            val_metric = val_metric["global"]
            test_per_class_df = test_metric["per_class"]
            test_metric = test_metric["global"]
            
            if val_long_seq_idx is not None and val_long_seq_metric is not None:
                val_long_seq_per_class_df = val_long_seq_metric["per_class"]
                val_long_seq_metric = val_long_seq_metric["global"]
            if test_long_seq_idx is not None and test_long_seq_metric is not None:
                test_long_seq_per_class_df = test_long_seq_metric["per_class"]
                test_long_seq_metric = test_long_seq_metric["global"]

        # Logging
        print(f"\nEpoch: {epoch:03d}, Average Loss: {ave_loss:.4f}")
        print(f"Validation: {val_metric}")
        print(f"Test:       {test_metric}")

        if test_subgroup_metrics is not None:
            print(f"Test-subgroups:       {test_subgroup_metrics}")
        if test_long_seq_metric is not None:
            print(f"Test-long:            {test_long_seq_metric}")

        # Check for improvement
        current_score = val_metric[eval_metric]
        if current_score > best_score:
            best_score = current_score
            if task_type == "binary":
                best_val_metric = val_metric
                best_test_metric = test_metric
                best_test_long_seq_metric = test_long_seq_metric
            else:
                best_val_metric = {"global": val_metric, "per_class": val_per_class_df}
                best_test_metric = {"global": test_metric, "per_class": test_per_class_df}
                best_test_long_seq_metric = {
                    "global": test_long_seq_metric,
                    "per_class": test_long_seq_per_class_df,
                } if test_long_seq_metric is not None else None

            # 只在 binary 任务下保留 subgroup metrics
            best_val_subgroup_metrics = val_subgroup_metrics if task_type == "binary" else None
            best_test_subgroup_metrics = test_subgroup_metrics if task_type == "binary" else None

            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        # Early stopping check
        if epochs_no_improve >= args["early_stop_patience"]:
            print(f"\nEarly stopping triggered after {epoch} epochs "
                  f"(no improvement for {args['early_stop_patience']} epochs).")
            break

    print("\nBest validation performance:")
    print(best_val_metric)
    print("Corresponding test performance:")
    print(best_test_metric)
    if best_test_long_seq_metric is not None:
        print("Corresponding test-long performance:")
        print(best_test_long_seq_metric)
    if best_test_subgroup_metrics is not None:
        print("Corresponding test-subgroup performance:")
        print(best_test_subgroup_metrics)

    return best_test_metric, best_test_long_seq_metric, best_test_subgroup_metrics

In [18]:
random.seed(42)
seeds = [random.randint(0, 2**32 - 1) for _ in range(5)]
print(seeds)

[2746317213, 1181241943, 958682846, 3163119785, 1812140441]


In [19]:
final_metrics, final_long_seq_metrics, final_subgroup_metrics = [], [], []

for seed in seeds:
    args["seed"] = seed
    set_random_seed(args["seed"])
    print(f"Training with seed: {args['seed']}")
    
    # Initialize model, optimizer, and loss function
    model = HBERT_Finetune(args)
    model.load_weight(torch.load(pretrained_weight_path, weights_only=True))
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=args["lr"])
    
    best_test_metric, best_test_long_seq_metric, best_test_subgroup_metrics = train_with_early_stopping(
        model, 
        train_dataloader, 
        val_dataloader, 
        test_dataloader,
        optimizer, 
        loss_fn, 
        device, 
        args,
        val_long_seq_idx,
        test_long_seq_idx,
        task_type=task_type,
        val_subgroup_labels=val_subgroup_labels,
        test_subgroup_labels=test_subgroup_labels)
    
    final_metrics.append(best_test_metric)
    final_long_seq_metrics.append(best_test_long_seq_metric)
    final_subgroup_metrics.append(best_test_subgroup_metrics)

[INFO] Random seed set to 2746317213
Training with seed: 2746317213


Running inference: 100%|██████████| 1590/1590 [00:02<00:00, 601.07it/s]



Epoch: 001, Average Loss: 0.6643
Validation: {'precision': 0.6093448940255074, 'recall': 0.7938500156862446, 'f1': 0.6894672249523978, 'auc': 0.6925072046403193, 'prauc': 0.6732212298237858}
Test:       {'precision': 0.6078336557045265, 'recall': 0.7822028624742309, 'f1': 0.6840816277298785, 'auc': 0.6963892266816138, 'prauc': 0.682680826036136}
Test-subgroups:       {'DIABETES': {'precision': 0.60183066361097, 'recall': 0.7788746298047495, 'f1': 0.6790017162467636, 'auc': 0.6869564950252527, 'prauc': 0.6916354016832243}, 'HYPERTENSION': {'precision': 0.59999999999738, 'recall': 0.7855917667193506, 'f1': 0.6803664224198346, 'auc': 0.6910130584410625, 'prauc': 0.6798496684113184}, 'CKD': {'precision': 0.5905612244822632, 'recall': 0.7678275290088254, 'f1': 0.6676279691202218, 'auc': 0.6748807609079115, 'prauc': 0.6607711445208604}, 'HEART_FAILURE': {'precision': 0.6433021806803482, 'recall': 0.7844254510846684, 'f1': 0.7068891691977013, 'auc': 0.7096639237280395, 'prauc': 0.70670102136

Running inference: 100%|██████████| 1590/1590 [00:02<00:00, 598.14it/s]



Epoch: 002, Average Loss: 0.6151
Validation: {'precision': 0.620957309183387, 'recall': 0.7530593034177815, 'f1': 0.6806579644146784, 'auc': 0.6989144149161506, 'prauc': 0.6880270411271778}
Test:       {'precision': 0.6192660550442934, 'recall': 0.756067205971512, 'f1': 0.6808629818786066, 'auc': 0.7010474510372084, 'prauc': 0.6848129201874108}
Test-subgroups:       {'DIABETES': {'precision': 0.6311475409784333, 'recall': 0.7549019607769127, 'f1': 0.6874999950337215, 'auc': 0.7055330178048231, 'prauc': 0.6897848040403265}, 'HYPERTENSION': {'precision': 0.6208352885939895, 'recall': 0.75599999999568, 'f1': 0.6817830406514763, 'auc': 0.7056812169312169, 'prauc': 0.6879146774346013}, 'CKD': {'precision': 0.6198910081659416, 'recall': 0.7374392220301874, 'f1': 0.6735751245612073, 'auc': 0.6914245046718059, 'prauc': 0.6809886987881423}, 'HEART_FAILURE': {'precision': 0.6227495908296011, 'recall': 0.7345559845488944, 'f1': 0.6740478249659553, 'auc': 0.69388203534545, 'prauc': 0.684101132350

...


Epoch: 010, Average Loss: 0.2408
Validation: {'precision': 0.655413271243727, 'recall': 0.7066206463736849, 'f1': 0.6800543510370235, 'auc': 0.7215977073962682, 'prauc': 0.7218533382850426}
Test:       {'precision': 0.6635649898206933, 'recall': 0.7100186683238643, 'f1': 0.6860063079451867, 'auc': 0.7305801510883523, 'prauc': 0.721647555298213}
Test-subgroups:       {'DIABETES': {'precision': 0.6575729068611706, 'recall': 0.708924949282871, 'f1': 0.6822840359960091, 'auc': 0.723478855336978, 'prauc': 0.6926371966540398}, 'HYPERTENSION': {'precision': 0.66005361929941, 'recall': 0.7091013824843946, 'f1': 0.6836989675102562, 'auc': 0.7347644531684012, 'prauc': 0.7135028505330127}, 'CKD': {'precision': 0.6712749615872308, 'recall': 0.717569786523521, 'f1': 0.6936507886453389, 'auc': 0.7357099791897621, 'prauc': 0.7216605058780341}, 'HEART_FAILURE': {'precision': 0.6690777576793031, 'recall': 0.7191448007704652, 'f1': 0.693208425913359, 'auc': 0.7358637847527651, 'prauc': 0.71932823749650

Running inference: 100%|██████████| 1590/1590 [00:02<00:00, 598.73it/s]



Epoch: 011, Average Loss: 0.2042
Validation: {'precision': 0.6634793337425556, 'recall': 0.6749294006881866, 'f1': 0.6691553846389751, 'auc': 0.7188591848603856, 'prauc': 0.7285388949768205}
Test:       {'precision': 0.6787630167223769, 'recall': 0.6692594897303383, 'f1': 0.6739777484056255, 'auc': 0.7302680721746937, 'prauc': 0.7263351255128587}
Test-subgroups:       {'DIABETES': {'precision': 0.6615541922222745, 'recall': 0.669079627707662, 'f1': 0.6652956248133703, 'auc': 0.7303736164722987, 'prauc': 0.711961962831328}, 'HYPERTENSION': {'precision': 0.6724039013157063, 'recall': 0.6770652801809529, 'f1': 0.6747265350076895, 'auc': 0.731888651569729, 'prauc': 0.7210262305563055}, 'CKD': {'precision': 0.6666666666555555, 'recall': 0.6666666666555555, 'f1': 0.6666666616555557, 'auc': 0.7267861111111111, 'prauc': 0.7076603273582782}, 'HEART_FAILURE': {'precision': 0.6763848396435727, 'recall': 0.6790243902372779, 'f1': 0.6777020397840726, 'auc': 0.7306846427258242, 'prauc': 0.725717743

Running inference: 100%|██████████| 1590/1590 [00:02<00:00, 596.89it/s]


Epoch: 012, Average Loss: 0.1723
Validation: {'precision': 0.6523297491019943, 'recall': 0.6852839661101812, 'f1': 0.6684009131341186, 'auc': 0.7126401476932157, 'prauc': 0.7137760733593518}
Test:       {'precision': 0.6640483383665738, 'recall': 0.6838830118211454, 'f1': 0.6738197374882875, 'auc': 0.7251440303316754, 'prauc': 0.7141144422696986}
Test-subgroups:       {'DIABETES': {'precision': 0.6562203228807577, 'recall': 0.6834817012790951, 'f1': 0.6695736384064351, 'auc': 0.7166213064161716, 'prauc': 0.7178641711228132}, 'HYPERTENSION': {'precision': 0.6633554083848601, 'recall': 0.6931949250248375, 'f1': 0.677946977514106, 'auc': 0.7375146159381183, 'prauc': 0.7256533786329764}, 'CKD': {'precision': 0.6593749999896973, 'recall': 0.7021630615523766, 'f1': 0.6800966912067092, 'auc': 0.7309325859238498, 'prauc': 0.719468957443194}, 'HEART_FAILURE': {'precision': 0.6483931947008659, 'recall': 0.677865612641523, 'f1': 0.6628019273632151, 'auc': 0.7205027762092979, 'prauc': 0.706191524

In [20]:
def topk_avg_performance_formatted(
    performances,
    long_seq_performances,
    subgroup_performances=None,
    k=5,
):
    """
    根据 overall 指标自动选 top-k 实验，并在这 k 个实验上计算：
      - overall 指标的均值 / 标准差
      - long-sequence 指标的均值 / 标准差
      - （可选）各 subgroup 指标的均值 / 标准差

    参数
    ----
    performances : list[dict]
        每个实验在“总体人群”上的指标，例如：
        [{"f1": 0.8, "auc": 0.9, "prauc": 0.7}, ...]
    long_seq_performances : list[dict]
        每个实验在 long-sequence 人群上的指标，长度与 performances 相同。
    subgroup_performances : list[dict[str, dict]] or None, 默认 None
        若不为 None，则形式为：
            [
                {
                    "DIABETES":     {"f1":..., "auc":..., "prauc":..., ...},
                    "HYPERTENSION": {...},
                    ...
                },
                {
                    "DIABETES":     {...},
                    "HYPERTENSION": {...},
                    ...
                },
                ...
            ]
        外层 list 长度 = 实验数 = len(performances)，
        每个 dict 的 key 为 subgroup 名（如 DIABETES），
        value 为该实验在该 subgroup 上的一组指标。
    k : int
        选取的 top-k 实验数量。

    返回
    ----
    results : dict
        {
            "overall_mean": {...},
            "overall_std": {...},
            "long_seq_mean": {...},
            "long_seq_std": {...},
            "subgroup": {
                subgroup_name: {
                    "mean": {...},
                    "std": {...}
                },
                ...
            } or None,
            "topk_idx": np.ndarray
        }
    """

    n = len(performances)
    if n == 0:
        raise ValueError("performances 为空")

    if len(long_seq_performances) != n:
        raise ValueError("long_seq_performances 长度与 performances 不一致")

    # =======================
    # 1. 根据 overall 选 top-k
    # =======================
    metrics_for_rank = ["f1", "auc", "prauc"]
    scores = {m: np.array([p[m] for p in performances]) for m in metrics_for_rank}
    # 越大越靠前：先按降序排序得到索引，再对索引排序得到名次（从 1 开始）
    ranks = {m: (-scores[m]).argsort().argsort() + 1 for m in metrics_for_rank}
    avg_ranks = np.mean(np.stack([ranks[m] for m in metrics_for_rank], axis=1), axis=1)
    topk_idx = np.argsort(avg_ranks)[:k]

    # =======================
    # 2. overall 均值 / 标准差
    # =======================
    metric_keys = list(performances[0].keys())

    overall_mean = {
        m: np.mean([performances[i][m] for i in topk_idx])
        for m in metric_keys
    }
    overall_std = {
        m: np.std([performances[i][m] for i in topk_idx], ddof=0)
        for m in metric_keys
    }

    # =======================
    # 3. long-seq 均值 / 标准差
    # =======================
    long_metric_keys = list(long_seq_performances[0].keys())
    long_seq_mean = {
        m: np.mean([long_seq_performances[i][m] for i in topk_idx])
        for m in long_metric_keys
    }
    long_seq_std = {
        m: np.std([long_seq_performances[i][m] for i in topk_idx], ddof=0)
        for m in long_metric_keys
    }

    # =======================
    # 4. subgroup（若提供）
    # =======================
    subgroup_results = None
    if subgroup_performances is not None:
        if len(subgroup_performances) != n:
            raise ValueError(
                f"subgroup_performances 长度 {len(subgroup_performances)} "
                f"与 performances 数量 {n} 不一致"
            )

        subgroup_results = {}
        # 从第一个实验的 dict 里拿到 subgroup 名称列表
        subgroup_names = list(subgroup_performances[0].keys())

        for subgroup_name in subgroup_names:
            # 取该 subgroup 对应的 metric dict 列表（按实验索引）
            sub_metric_dicts = [subgroup_performances[i][subgroup_name] for i in topk_idx]

            sub_metric_keys = list(sub_metric_dicts[0].keys())
            sub_mean = {
                m: np.mean([d[m] for d in sub_metric_dicts])
                for m in sub_metric_keys
            }
            sub_std = {
                m: np.std([d[m] for d in sub_metric_dicts], ddof=0)
                for m in sub_metric_keys
            }
            subgroup_results[subgroup_name] = {"mean": sub_mean, "std": sub_std}

    # =======================
    # 5. 打印结果
    # =======================
    print("=== Overall (Top-k) ===")
    for m in overall_mean.keys():
        print(f"{m}: {overall_mean[m]:.4f} ± {overall_std[m]:.4f}")

    print("\n=== Long-sequence (Top-k) ===")
    for m in long_seq_mean.keys():
        print(f"{m}: {long_seq_mean[m]:.4f} ± {long_seq_std[m]:.4f}")

    if subgroup_results is not None:
        print("\n=== Subgroup (Top-k) ===")
        for subgroup_name, res in subgroup_results.items():
            print(f"\n[{subgroup_name}]")
            for m in res["mean"].keys():
                print(f"{m}: {res['mean'][m]:.4f} ± {res['std'][m]:.4f}")

In [21]:
def print_per_class_performance(dfs, col_name="prauc"):
    """
    输入一个 DataFrame 列表，对每个疾病在所有表格的指定列计算 mean ± std 并打印。

    参数:
        dfs (list[pd.DataFrame]): 多个表格组成的列表
        col_name (str): 要计算的指标列名 (默认: "prauc")
    """
    # 拼接所有表格
    all_values = pd.concat(dfs, axis=0)

    # 按疾病分组，计算 mean 和 std
    grouped = all_values.groupby(all_values.index)[col_name].agg(["mean", "std"])

    # 打印
    for disease, row in grouped.iterrows():
        mean_val = row["mean"] * 100
        std_val = row["std"] * 100
        print(f"{disease}: {mean_val:.2f} ± {std_val:.2f}")

In [22]:
if task_type == "binary":
    topk_avg_performance_formatted(final_metrics, final_long_seq_metrics, final_subgroup_metrics)
else:
    final_metrics_global = [metrics["global"] for metrics in final_metrics]
    final_metrics_per_class = [metrics["per_class"] for metrics in final_metrics]
    final_long_seq_metrics_global = [metrics["global"] for metrics in final_long_seq_metrics]
    final_long_seq_metrics_per_class = [metrics["per_class"] for metrics in final_long_seq_metrics]
    topk_avg_performance_formatted(final_metrics_global, final_long_seq_metrics_global)
    print("\nPer-class performance, all patients:")
    print_per_class_performance(final_metrics_per_class, col_name="prauc")
    print("\nPer-class performance, long seq:")
    print_per_class_performance(final_long_seq_metrics_per_class, col_name="prauc")

=== Overall (Top-k) ===
precision: 0.6532 ± 0.0253
recall: 0.7351 ± 0.0259
f1: 0.6908 ± 0.0043
auc: 0.7286 ± 0.0168
prauc: 0.7179 ± 0.0188

=== Long-sequence (Top-k) ===
precision: 0.6433 ± 0.0189
recall: 0.7246 ± 0.0445
f1: 0.6802 ± 0.0115
auc: 0.7204 ± 0.0169
prauc: 0.7100 ± 0.0320

=== Subgroup (Top-k) ===

[DIABETES]
precision: 0.6540 ± 0.0280
recall: 0.7356 ± 0.0288
f1: 0.6913 ± 0.0072
auc: 0.7291 ± 0.0220
prauc: 0.7204 ± 0.0170

[HYPERTENSION]
precision: 0.6495 ± 0.0284
recall: 0.7365 ± 0.0259
f1: 0.6892 ± 0.0072
auc: 0.7270 ± 0.0191
prauc: 0.7135 ± 0.0192

[CKD]
precision: 0.6493 ± 0.0354
recall: 0.7315 ± 0.0196
f1: 0.6869 ± 0.0139
auc: 0.7269 ± 0.0296
prauc: 0.7145 ± 0.0301

[HEART_FAILURE]
precision: 0.6588 ± 0.0195
recall: 0.7319 ± 0.0283
f1: 0.6929 ± 0.0140
auc: 0.7272 ± 0.0140
prauc: 0.7161 ± 0.0158

[CAD]
precision: 0.6507 ± 0.0197
recall: 0.7341 ± 0.0313
f1: 0.6890 ± 0.0069
auc: 0.7286 ± 0.0068
prauc: 0.7196 ± 0.0105

[COPD]
precision: 0.6535 ± 0.0161
recall: 0.7500 ± 0.0